In [ ]:

# =============================================================================
# Reproduce all paper forecast figures by loading:
#   - MCMC posteriors directly from .h5 backend files  (no large pickles needed)
#   - GGL catalog FITS files for the pairing-quality figures
#
# This notebook replaces the need to have the large pdspl_samples_with_pairs.pkl
# and forecast_scenarios_*.pkl files.  Everything is reconstructed from the
# small files that are tracked on GitHub.
#
# Figures produced
# ----------------
# Fig 3  — Unpaired GGL population corner plot
# Fig 4  — β_E vs D with linear scatter fit
# Fig 4b — Pairing scatter (lens 1 vs lens 2)
# Fig 5  — Relative-difference corner plot
# Table 2— LaTeX pairing statistics
# Fig 6  — LSST Y1 vs Y10 forecast
# Fig 7  — LSST Y10 vs 4MOST forecast
# Fig 8  — DSPL vs PDSPL Y10 (±Ωm prior)
# Fig 9  — β_E vs D photo-z vs spec-z (Appendix B)
# Fig 10 — Photo-z vs spec-z forecast (Appendix B)
# Fig 11 — Free-scatter posteriors (Appendix C)
# Fig 12 — Free vs fixed scatter comparison (Appendix C)
# =============================================================================

# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import os
import numpy as np
import matplotlib.pyplot as plt
import emcee
from tqdm import tqdm
from astropy.cosmology import FlatLambdaCDM
from astropy import units as u

from pdspl_utils.data_utils import preprocess_ggl_table
from pdspl_utils.pairing import (
    get_kdtree_pairs,
    get_pairs_table_PDSPL,
    inject_observational_errors,
    compute_dissimilarity,
    eval_scatter_model,
    plot_beta_E_vs_D_MC,
    plot_dataset_corner,
    plot_reldiff_corner,
    plot_pairing_scatter,
    generate_latex_summary_table,
)
from pdspl_utils.plotting import plot_dspl_corner

import warnings
warnings.filterwarnings("ignore")

# %load_ext autoreload
# %autoreload 2

FIGURE_DIR          = "../figures/all_paper_figures"
POSTERIOR_FIXED_DIR = "../data/posteriors/forecast_w0waCDM_fixed_scatter"
POSTERIOR_FREE_DIR  = "../data/posteriors/forecast_w0waCDM_free_scatter"
os.makedirs(FIGURE_DIR, exist_ok=True)


# ── Cell 2: Colour palette, truth values, shared plot config ─────────────────
C_SKYBLUE   = "#56B4E9"   # DSPL
C_ORANGE    = "#E69F00"   # LSST Y1
C_GREEN     = "#009E73"   # LSST Y10
C_VERMILION = "#D55E00"   # free-scatter with prior
C_PURPLE    = "#CC79A7"   # 4MOST spec-z
C_BLUE      = "#0072B2"   # 4MOST spec-z + σv / DSPL w/ prior
C_BLACK     = "#000000"   # Y10 with Ωm prior

accessible_colors = {
    "lsst_y1":                   C_ORANGE,
    "lsst_y10":                  C_GREEN,
    "lsst_y10_baseline":         C_GREEN,
    "lsst_4most_spec-z":         C_PURPLE,
    "lsst_4most_spec-z_sigma_v": C_BLUE,
    "DSPL":                      C_SKYBLUE,
    "lsst_y10_om_prior":         C_BLACK,
    "DSPL_om_prior":             C_BLUE,
    "lsst_y10_zD_spec":          C_SKYBLUE,
    "lsst_y10_all_spec":         C_PURPLE,
}
accessible_markers = {
    "lsst_y1":                   "^",
    "lsst_y10":                  "o",
    "lsst_4most_spec-z":         "s",
    "lsst_4most_spec-z_sigma_v": "D",
    "lsst_y10_photo_z":          "o",
    "lsst_y10_spec_z":           "s",
}

truth = {
    "h0": 70.0, "om": 0.3, "w0": -1.0, "wa": 0.0,
    "lambda_int": 1.0, "lambda_sigma": 0.05,
    "gamma_pl":   2.0, "gamma_sigma":  0.16,
}
fixed_params = {"h0": 70.0}
cosmo_true   = FlatLambdaCDM(H0=70, Om0=0.3)

latex_labels = {
    "om":           r"$\Omega_{\rm m}$",
    "w0":           r"$w_0$",
    "wa":           r"$w_a$",
    "lambda_int":   r"$\overline{\lambda}_{\rm MST}$",
    "lambda_sigma": r"$\sigma({\lambda}_{\rm MST})$",
    "gamma_pl":     r"$\overline{\gamma}_{\rm pl}$",
    "gamma_sigma":  r"$\sigma({\gamma}_{\rm pl})$",
    "beta_c0":      r"$\sigma_{\beta_{\rm E},\rm \mathcal{D}}^{(0)}$",
    "beta_c1":      r"$\sigma_{\beta_{\rm E},\rm \mathcal{D}}^{(1)}$",
}
custom_ranges_7 = [          # 7-parameter fixed-scatter space
    (0, 1),                  # om
    (-2, 0),                 # w0
    (-3, 3),                 # wa
    (0.8, 1.2),              # lambda_int
    (0.0, 0.18),             # lambda_sigma
    (1.8, 2.2),              # gamma_pl
    (0.0, 0.32),             # gamma_sigma
]
custom_ranges_9 = custom_ranges_7 + [(0, 0.25), (0, 2.0)]   # + beta_c0, beta_c1


# ── Cell 3: Helper — load flat samples from an emcee .h5 backend ─────────────
def load_h5(path, n_burn=500):
    """
    Load flat posterior samples from an emcee HDF5 backend.

    Parameters
    ----------
    path : str
    n_burn : int   Burn-in steps to discard.

    Returns
    -------
    numpy.ndarray or None
        Shape (N_samples, N_params).  None if the file is absent.
    """
    if not os.path.exists(path):
        print(f"  [not found] {path}")
        return None
    try:
        reader = emcee.backends.HDFBackend(path, read_only=True)
        return reader.get_chain(discard=n_burn, thin=1, flat=True)
    except Exception as e:
        print(f"  [error reading {path}] {e}")
        return None


def scenario(name, color, h5_path, n_burn=500):
    """Return a minimal plotting-ready scenario dict."""
    return {"name": name, "color": color, "samples": load_h5(h5_path, n_burn)}


# ── Cell 4: Load all fixed-scatter posteriors ─────────────────────────────────
# Parameter order in .h5 chains (7 free params, h0 + beta_c{0,1,2} fixed):
#   om, w0, wa, lambda_int, lambda_sigma, gamma_pl, gamma_sigma

print("Loading fixed-scatter posteriors …")
P = POSTERIOR_FIXED_DIR    # shorthand

fixed_scenarios = {
    # ── Main text ──────────────────────────────────────────────────────
    "lsst_y1": scenario(
        "PDSPL (LSST Y1)",    C_ORANGE,  f"{P}/lsst_y1.h5"),
    "lsst_y10": scenario(
        "PDSPL (LSST Y10)",   C_GREEN,   f"{P}/lsst_y10.h5"),
    "lsst_4most_spec-z": scenario(
        r"PDSPL (4MOST $z^{\rm spec}$)",             C_PURPLE, f"{P}/lsst_4most_spec-z.h5"),
    "lsst_4most_spec-z_sigma_v": scenario(
        r"PDSPL (4MOST $z^{\rm spec}$ + $\sigma_{v,D}$)", C_BLUE, f"{P}/lsst_4most_spec-z_sigma_v.h5"),
    "DSPL": scenario(
        "DSPL (500 lenses)",  C_SKYBLUE, f"{P}/DSPL.h5"),

    # ── With Ωm prior ──────────────────────────────────────────────────
    "lsst_y1_om_prior": scenario(
        r"PDSPL (LSST Y1) + $\Omega_m$ Prior",  C_ORANGE, f"{P}/lsst_y1_om_prior.h5"),
    "lsst_y10_om_prior": scenario(
        r"PDSPL (LSST Y10) + $\Omega_m$ Prior", C_BLACK,  f"{P}/lsst_y10_om_prior.h5"),
    "lsst_4most_spec-z_om_prior": scenario(
        r"PDSPL (4MOST $z^{\rm spec}$) + $\Omega_m$ Prior",             C_PURPLE, f"{P}/lsst_4most_spec-z_om_prior.h5"),
    "lsst_4most_spec-z_sigma_v_om_prior": scenario(
        r"PDSPL (4MOST $z^{\rm spec}$ + $\sigma_{v,D}$) + $\Omega_m$ Prior", C_BLUE,   f"{P}/lsst_4most_spec-z_sigma_v_om_prior.h5"),
    "DSPL_om_prior": scenario(
        r"DSPL (500 lenses) + $\Omega_m$ Prior", C_BLUE,   f"{P}/DSPL_om_prior.h5"),

    # ── Appendix B (spec-z experiment) ─────────────────────────────────
    "lsst_y10_zD_spec": scenario(
        r"LSST Y10 ($z_D^{\rm spec}$, $z_S^{\rm phot}$)", C_SKYBLUE, f"{P}/lsst_y10_zD_spec.h5"),
    "lsst_y10_all_spec": scenario(
        r"LSST Y10 ($z_D^{\rm spec}$, $z_S^{\rm spec}$)",  C_PURPLE,  f"{P}/lsst_y10_all_spec.h5"),
}

# ── Load free-scatter posteriors (Appendix C) ─────────────────────────────────
# Parameter order (9 free params, h0 + beta_c2 fixed):
#   om, w0, wa, lambda_int, lambda_sigma, gamma_pl, gamma_sigma, beta_c0, beta_c1

print("Loading free-scatter posteriors …")
PF = POSTERIOR_FREE_DIR

free_scenarios = {
    "lsst_y10": scenario(
        "PDSPL (LSST Y10) (Free Scat.)",                     C_ORANGE,    f"{PF}/lsst_y10.h5"),
    "lsst_y10_om_prior": scenario(
        r"PDSPL (LSST Y10) + $\Omega_m$ Prior (Free Scat.)", C_VERMILION, f"{PF}/lsst_y10_om_prior.h5"),
}

# Summary
n_found   = sum(1 for s in {**fixed_scenarios, **free_scenarios}.values() if s["samples"] is not None)
n_missing = sum(1 for s in {**fixed_scenarios, **free_scenarios}.values() if s["samples"] is None)
print(f"  Loaded {n_found} chains; {n_missing} not found (those figures will be skipped).")


# ── Cell 5: Re-run pairing (needed for Figs 3, 4, 4b, 5, Table 2) ────────────
#
# Running 100 MC realisations takes ~2 min.  Set N_realizations=1 if you only
# want the forecast figures (Figs 6–12) and don't need the MC-averaged curve
# in Fig 4 right panel.

print("\nLoading GGL catalogs and running pairing …")
GGL_Y10  = preprocess_ggl_table("../data/GGL_Catalogs/GGL_20000.0_SQDEG_LSSTY10_SNR_20_seeing_0.5_contrast_02.fits")
GGL_4M   = preprocess_ggl_table("../data/GGL_Catalogs/GGL_20000.0_SQDEG_4MOST_SNR_20_seeing_0.2_contrast_no.fits")

GGL_Y1   = GGL_Y10.copy();  GGL_Y1.sort("snr_i",   reverse=True); GGL_Y1  = GGL_Y1[:int(len(GGL_Y10)/np.sqrt(10))]
GGL_4Mz  = GGL_4M.copy();   GGL_4Mz.sort("mag_D_r");              GGL_4Mz = GGL_4Mz[:10000]
GGL_4Mzs = GGL_4M.copy();   GGL_4Mzs.sort("mag_D_r");             GGL_4Mzs= GGL_4Mzs[:5000]

pdspl_samples = {
    "lsst_y1":                   {"table": GGL_Y1,   "name": "LSST Y1",                                        "color": C_ORANGE},
    "lsst_y10":                  {"table": GGL_Y10,  "name": "LSST Y10",                                       "color": C_GREEN},
    "lsst_4most_spec-z":         {"table": GGL_4Mz,  "name": r"4MOST ($z^{\rm spec}$)",                        "color": C_PURPLE},
    "lsst_4most_spec-z_sigma_v": {"table": GGL_4Mzs, "name": r"4MOST ($z^{\rm spec}$ + $\sigma_{v,D}$)",      "color": C_BLUE},
}

_base_keys   = ["z_D","R_e_arcsec","flux_D_i","flux_ratio_D_gr","flux_ratio_D_ri"]
_base_dissim = ["rel_diff_z_D","rel_diff_R_e_arcsec",
                "rel_diff_flux_D_i","rel_diff_flux_ratio_D_gr","rel_diff_flux_ratio_D_ri"]
_base_err    = {
    "sigma_v_D":       10.0,
    "R_e_arcsec":      lambda t: 0.05  * t["R_e_arcsec"],
    "flux_D_i":        lambda t: 0.01  * t["flux_D_i"],
    "flux_ratio_D_gr": lambda t: 0.014 * t["flux_ratio_D_gr"],
    "flux_ratio_D_ri": lambda t: 0.014 * t["flux_ratio_D_ri"],
}
for key, s in pdspl_samples.items():
    s["pairing_keys"]      = _base_keys + (["sigma_v_D"] if "sigma_v" in key else [])
    s["dissimilarity_keys"] = _base_dissim + (["rel_diff_sigma_v_D"] if "sigma_v" in key else [])
    s["error_config"]      = {**_base_err,
        "z_D": (lambda t: 0.03*(1+t["z_D"])) if key in ("lsst_y1","lsst_y10") else 1e-4}

N_realizations = 100   # set to 1 for a quick test
mc_results = {k: {"binned_dissim":[],"binned_scatter":[],"num_pairs":[],"scatter_in_beta_E":[]}
              for k in pdspl_samples}

for _ in tqdm(range(N_realizations), desc="MC pairing"):
    for sk, s in pdspl_samples.items():
        te  = inject_observational_errors(s["table"], s["error_config"])
        idx, _ = get_kdtree_pairs(te, s["pairing_keys"], use_log_metric=True)
        pt  = get_pairs_table_PDSPL(s["table"], idx, cosmo_true)
        pte = get_pairs_table_PDSPL(te,          idx, cosmo_true)
        pte["dissimilarity"] = compute_dissimilarity(pte, s["dissimilarity_keys"], method="rms")
        s["pairs_analysis"] = {
            "pairs_table": pt, "pairs_table_with_errors": pte,
            "num_lenses": len(s["table"]), "num_pairs": len(pt),
            "scatter_in_beta_E": np.std(pt["rel_diff_beta_E"]), "pair_indices": idx,
        }
        mc_results[sk]["num_pairs"].append(len(pt))
        mc_results[sk]["scatter_in_beta_E"].append(np.std(pt["rel_diff_beta_E"]))
        mask = (pte["dissimilarity"] < 0.15) & np.isfinite(pt["rel_diff_beta_E"])
        dm, rm = pte["dissimilarity"][mask], pt["rel_diff_beta_E"][mask]
        pcts = np.percentile(dm, np.arange(0,101,10)); dig = np.digitize(dm, pcts)
        rd, rs = [], []
        for i in range(1, len(pcts)):
            bm = dig == i
            rd.append(np.median(dm[bm])  if np.any(bm) else np.nan)
            rs.append(np.nanstd(rm[bm])  if np.any(bm) else np.nan)
        mc_results[sk]["binned_dissim"].append(rd)
        mc_results[sk]["binned_scatter"].append(rs)

for sk, s in pdspl_samples.items():
    s["pairs_analysis"]["mean_num_pairs"]         = np.nanmean(mc_results[sk]["num_pairs"])
    s["pairs_analysis"]["mean_scatter_in_beta_E"] = np.nanmean(mc_results[sk]["scatter_in_beta_E"])

print("Pairing done.")




# ── Cell 6: Re-run spec-z pairing (needed for Figs 9, 10) ────────────────────
print("\nSpec-z pairing for Appendix B …")

pdspl_expt = {
    "lsst_y10_photo_z": {"table": GGL_Y10, "name": r"LSST Y10 ($z_D^{\rm photo}$)", "color": C_GREEN},
    "lsst_y10_spec_z":  {"table": GGL_Y10, "name": r"LSST Y10 ($z_D^{\rm spec}$)",  "color": C_BLUE},
}
for sk, s in pdspl_expt.items():
    s["pairing_keys"]      = _base_keys
    s["dissimilarity_keys"] = _base_dissim
    s["error_config"]      = {**_base_err,
        "z_D": (lambda t: 0.03*(1+t["z_D"])) if "photo" in sk else 1e-4}

mc_expt = {k: {"binned_dissim":[],"binned_scatter":[],"num_pairs":[],"scatter_in_beta_E":[]}
           for k in pdspl_expt}

for _ in tqdm(range(100), desc="MC spec-z pairing"):
    for sk, s in pdspl_expt.items():
        te  = inject_observational_errors(s["table"], s["error_config"])
        idx, _ = get_kdtree_pairs(te, s["pairing_keys"], use_log_metric=True)
        pt  = get_pairs_table_PDSPL(s["table"], idx, cosmo_true)
        pte = get_pairs_table_PDSPL(te,          idx, cosmo_true)
        pte["dissimilarity"] = compute_dissimilarity(pte, s["dissimilarity_keys"], method="rms")
        s["pairs_analysis"] = {
            "pairs_table": pt, "pairs_table_with_errors": pte,
            "num_lenses": len(s["table"]), "num_pairs": len(pt),
            "scatter_in_beta_E": np.std(pt["rel_diff_beta_E"]), "pair_indices": idx,
        }
        mc_expt[sk]["num_pairs"].append(len(pt))
        mc_expt[sk]["scatter_in_beta_E"].append(np.std(pt["rel_diff_beta_E"]))
        mask = (pte["dissimilarity"] < 0.1) & np.isfinite(pt["rel_diff_beta_E"])
        dm, rm = pte["dissimilarity"][mask], pt["rel_diff_beta_E"][mask]
        pcts = np.percentile(dm, np.arange(0,101,10)); dig = np.digitize(dm, pcts)
        rd, rs = [], []
        for i in range(1, len(pcts)):
            bm = dig == i
            rd.append(np.median(dm[bm])  if np.any(bm) else np.nan)
            rs.append(np.nanstd(rm[bm])  if np.any(bm) else np.nan)
        mc_expt[sk]["binned_dissim"].append(rd)
        mc_expt[sk]["binned_scatter"].append(rs)

print("Spec-z pairing done.")

# Assemble photo-z vs spec-z scenario dict for Fig 10
photo_specz_scenarios = {
    "lsst_y10_baseline": {
        "name":    r"PDSPL LSST Y10 ($z_D^{\rm phot}$, $z_S^{\rm phot}$)",
        "color":   C_GREEN,
        "samples": fixed_scenarios["lsst_y10"]["samples"],
    },
    "lsst_y10_zD_spec": {
        "name":    fixed_scenarios["lsst_y10_zD_spec"]["name"],
        "color":   C_SKYBLUE,
        "samples": fixed_scenarios["lsst_y10_zD_spec"]["samples"],
    },
    "lsst_y10_all_spec": {
        "name":    fixed_scenarios["lsst_y10_all_spec"]["name"],
        "color":   C_PURPLE,
        "samples": fixed_scenarios["lsst_y10_all_spec"]["samples"],
    },
}


# ── Cell 7: Figure 3 — Unpaired GGL population corner plot ───────────────────
print("\nFig 3 …")
fig3 = plot_dataset_corner(
    pdspl_samples,
    samples_to_plot=["lsst_y1","lsst_y10","lsst_4most_spec-z","lsst_4most_spec-z_sigma_v"],
    key_list=["z_D","z_S","theta_E","sigma_v_D","e_mass_D","mag_D_i","mag_S_i_lensed"],
    key_latex_labels={
        "z_D": r"$z_D$", "z_S": r"$z_S$", "theta_E": r"$\theta_E$",
        "sigma_v_D": r"$\sigma_{v,D}$", "e_mass_D": r"$q_{mass,D}$",
        "mag_D_i": r"$m_{D,i}$", "mag_S_i_lensed": r"$m_{S,i}^{lensed}$",
    },
    plot_ranges=[(0,2.5),(0,5),(0,2.5),(150,450),(0,0.4),(17,27),(18,26)],
    custom_colors_dict=accessible_colors,
    save_path=f"{FIGURE_DIR}/slsim_corner_GGL_all_samples_v2.pdf",
)
plt.close(fig3); print("  → Fig 3 saved")


# ── Cell 8: Figure 4 — β_E vs D (linear fit) ─────────────────────────────────
print("Fig 4 …")
fig4 = plot_beta_E_vs_D_MC(
    pdspl_samples, mc_results,
    fit_type="linear",
    custom_colors_dict=accessible_colors,
    custom_markers_dict=accessible_markers,
    save_path=f"{FIGURE_DIR}/beta_E_vs_D_MC.png",
)
plt.close(fig4); print("  → Fig 4 saved")

# Pull fit coefficients for use in free-scatter truth lines
fit_coeffs = {k: {"coeffs":   s["pairs_analysis"]["scatter_vs_dissimilarity_fit_coeffs"],
                  "fit_type": s["pairs_analysis"]["scatter_vs_dissimilarity_fit_type"]}
              for k, s in pdspl_samples.items()
              if "scatter_vs_dissimilarity_fit_coeffs" in s["pairs_analysis"]}

# ── Cell 9: Table 2 — LaTeX pairing statistics ────────────────────────────────
print("\nTable 2 (LaTeX):")
generate_latex_summary_table(
    pdspl_samples,
    ["lsst_y1","lsst_y10","lsst_4most_spec-z","lsst_4most_spec-z_sigma_v"],
)


# ── Cell 10: Figure 4b — Pairing scatter (lens 1 vs lens 2) ──────────────────
print("\nFig 4b …")
fig4b = plot_pairing_scatter(
    pdspl_samples=pdspl_samples,
    samples_to_plot=["lsst_y1","lsst_y10","lsst_4most_spec-z","lsst_4most_spec-z_sigma_v"],
    pairing_param_keys=["z_D","R_e_arcsec","flux_D_i","flux_ratio_D_gr","flux_ratio_D_ri","sigma_v_D"],
    pairing_param_labels={
        "z_D":             r"$z_D$",
        "R_e_arcsec":      r"$R_e$ [arcsec]",
        "flux_D_i":        r"$f_{D,i}$",
        "flux_ratio_D_gr": r"$f_{D,g}/f_{D,r}$",
        "flux_ratio_D_ri": r"$f_{D,r}/f_{D,i}$",
        "sigma_v_D":       r"$\sigma_{v,D}$ [km/s]",
    },
    pairing_hist_labels={
        "z_D":             r"$\Delta z_D / \langle z_D \rangle$",
        "R_e_arcsec":      r"$\Delta R_e / \langle R_e \rangle$",
        "flux_D_i":        r"$\Delta f_{D,i} / \langle f_{D,i} \rangle$",
        "flux_ratio_D_gr": r"$\Delta (f_{D,g}/f_{D,r}) / \langle f_{D,g}/f_{D,r} \rangle$",
        "flux_ratio_D_ri": r"$\Delta (f_{D,r}/f_{D,i}) / \langle f_{D,r}/f_{D,i} \rangle$",
        "sigma_v_D":       r"$\Delta \sigma_{v,D} / \langle \sigma_{v,D} \rangle$",
    },
    custom_colors_dict=accessible_colors,
    custom_markers_dict=accessible_markers,
    n_scatter_points=2000,
    reldiff_range=(-0.25, 0.25),
    save_path=f"{FIGURE_DIR}/pairing_scatter_lens1_vs_lens2.pdf",
)
fig4b.tight_layout()
plt.close(fig4b); print("  → Fig 4b saved")


# ── Cell 11: Figure 5 — Relative-difference corner plot ──────────────────────
print("Fig 5 …")
fig5 = plot_reldiff_corner(
    pdspl_samples,
    samples_to_plot=["lsst_y1","lsst_y10","lsst_4most_spec-z","lsst_4most_spec-z_sigma_v"],
    key_list=["rel_diff_z_D","rel_diff_sigma_v_D","rel_diff_beta_E",
              "rel_diff_gamma_pl","rel_diff_mag_D_i","rel_diff_color_D_gr"],
    key_latex_labels={
        "rel_diff_z_D":        r"$\Delta z_D / z_D$",
        "rel_diff_sigma_v_D":  r"$\Delta \sigma_{v,D} / \sigma_{v,D}$",
        "rel_diff_beta_E":     r"$\Delta \beta_E / \beta_E$",
        "rel_diff_gamma_pl":   r"$\Delta \gamma_{\rm pl} / \gamma_{\rm pl}$",
        "rel_diff_mag_D_i":    r"$\Delta m_{D,i} / m_{D,i}$",
        "rel_diff_color_D_gr": r"$\Delta c_{D,g-r} / c_{D,g-r}$",
    },
    custom_ranges={
        "rel_diff_z_D":        (-0.2, 0.2), "rel_diff_sigma_v_D": (-0.5, 0.5),
        "rel_diff_beta_E":     (-0.5, 0.5), "rel_diff_gamma_pl":  (-0.4, 0.4),
        "rel_diff_mag_D_i":    (-0.3, 0.3), "rel_diff_color_D_gr":(-0.3, 0.3),
    },
    custom_colors_dict=accessible_colors,
    save_path=f"{FIGURE_DIR}/pairing_reldiff_corner_all_samples.pdf",
)
plt.close(fig5); print("  → Fig 5 saved")


# ── Cell 12: Figures 6, 7, 8 — Fixed-scatter forecast corner plots ────────────
_ls = {
    "lsst_y1":                   "--",
    "lsst_y10":                  "-",
    "lsst_4most_spec-z":         "-.",
    "lsst_4most_spec-z_sigma_v": ":",
    "DSPL":                      "--",
    "DSPL_om_prior":             "--",
    "lsst_y10_om_prior":         "-",
}

print("Fig 6: LSST Y1 vs Y10 …")
fig6 = plot_dspl_corner(
    fixed_scenarios, truth, fixed_params,
    scenarios_to_plot=["lsst_y1","lsst_y10"],
    custom_ranges=custom_ranges_7, latex_labels=latex_labels, figsize=(14,14),
    show_multiple_titles=True,
    custom_colors_dict=accessible_colors, custom_linestyles_dict=_ls,
    save_path=f"{FIGURE_DIR}/pdspls_lsst_y1_y10_forecast_w0waCDM_fixed_scatter.pdf",
)
plt.close(fig6); print("  → Fig 6 saved")

print("Fig 7: LSST Y10 vs 4MOST …")
fig7 = plot_dspl_corner(
    fixed_scenarios, truth, fixed_params,
    scenarios_to_plot=["lsst_4most_spec-z_sigma_v","lsst_4most_spec-z","lsst_y10"],
    custom_ranges=custom_ranges_7, latex_labels=latex_labels, figsize=(14,14),
    show_multiple_titles=True,
    custom_colors_dict=accessible_colors, custom_linestyles_dict=_ls,
    save_path=f"{FIGURE_DIR}/pdspls_lsst_y10_vs_4MOST_forecast_w0waCDM_fixed_scatter.pdf",
)
plt.close(fig7); print("  → Fig 7 saved")

print("Fig 8: DSPL vs PDSPL (±Ωm prior) …")
fig8 = plot_dspl_corner(
    fixed_scenarios, truth, fixed_params,
    scenarios_to_plot=["DSPL","lsst_y10","DSPL_om_prior","lsst_y10_om_prior"],
    custom_ranges=custom_ranges_7, latex_labels=latex_labels, figsize=(14,14),
    show_multiple_titles=True,
    custom_colors_dict=accessible_colors, custom_linestyles_dict=_ls,
    save_path=f"{FIGURE_DIR}/pdspl_vs_dspl_forecast_w0waCDM_fixed_scatter.pdf",
)
plt.close(fig8); print("  → Fig 8 saved")


# ── Cell 13: Figure 9 — β_E vs D photo-z vs spec-z (Appendix B) ──────────────
print("Fig 9 …")
fig9 = plot_beta_E_vs_D_MC(
    pdspl_expt, mc_expt,
    fit_type="linear",
    custom_colors_dict={"lsst_y10_photo_z": C_GREEN, "lsst_y10_spec_z": C_BLUE},
    custom_markers_dict={"lsst_y10_photo_z": "o", "lsst_y10_spec_z": "s"},
    save_path=f"{FIGURE_DIR}/beta_E_vs_D_LSST_Y10_photo_vs_spec-z.png",
)
plt.close(fig9); print("  → Fig 9 saved")


# ── Cell 14: Figure 10 — Photo-z vs spec-z forecast (Appendix B) ─────────────
print("Fig 10 …")
_have_specz = all(photo_specz_scenarios[k]["samples"] is not None
                  for k in ["lsst_y10_baseline","lsst_y10_zD_spec","lsst_y10_all_spec"])
if _have_specz:
    fig10 = plot_dspl_corner(
        photo_specz_scenarios, truth, fixed_params,
        scenarios_to_plot=["lsst_y10_baseline","lsst_y10_zD_spec","lsst_y10_all_spec"],
        custom_ranges=custom_ranges_7, latex_labels=latex_labels, figsize=(14,14),
        show_multiple_titles=True,
        custom_colors_dict={
            "lsst_y10_baseline": C_GREEN,
            "lsst_y10_zD_spec":  C_SKYBLUE,
            "lsst_y10_all_spec": C_PURPLE,
        },
        custom_linestyles_dict={
            "lsst_y10_baseline": "-",
            "lsst_y10_zD_spec":  "--",
            "lsst_y10_all_spec": "-.",
        },
        save_path=f"{FIGURE_DIR}/lsst_y10_photoz_vs_specz_forecast.pdf",
    )
    plt.close(fig10); print("  → Fig 10 saved")
else:
    print("  [skip] spec-z posteriors not found")


# ── Cell 15: Figure 11 — Free-scatter posteriors (Appendix C) ────────────────
print("Fig 11 …")
_have_free = all(free_scenarios[k]["samples"] is not None for k in free_scenarios)

if _have_free:
    # Truth lines for the scatter params use the calibrated linear fit coefficients
    _fc = fit_coeffs.get("lsst_y10", {})
    custom_truth_free = truth.copy()
    if _fc:
        # linear coeffs: [slope, intercept] → intercept = σ^(0), slope = σ^(1)
        custom_truth_free["beta_c0"] = _fc["coeffs"][1]   # intercept = σ^(0)
        custom_truth_free["beta_c1"] = _fc["coeffs"][0]   # slope     = σ^(1)
    else:
        custom_truth_free["beta_c0"] = None
        custom_truth_free["beta_c1"] = None

    fig11 = plot_dspl_corner(
        free_scenarios, custom_truth_free, fixed_params,
        scenarios_to_plot=["lsst_y10","lsst_y10_om_prior"],
        custom_ranges=custom_ranges_9,
        latex_labels=latex_labels, figsize=(14,14),
        show_multiple_titles=True,
        save_path=f"{FIGURE_DIR}/free_scatter_lsst_y10_forecast_w0waCDM.pdf",
        titles_fontsize=12,
    )
    plt.close(fig11); print("  → Fig 11 saved")

    # ── Figure 12 — Free vs fixed comparison ─────────────────────────────────
    print("Fig 12 …")
    merged = {}
    for k in ["lsst_y10","lsst_y10_om_prior"]:
        if fixed_scenarios[k]["samples"] is not None:
            merged[f"fixed_{k}"] = {
                "name":    fixed_scenarios[k]["name"] + " (Fixed Scat.)",
                "color":   C_GREEN if "om" not in k else C_BLACK,
                "samples": fixed_scenarios[k]["samples"][:, :7],
            }
        if free_scenarios[k]["samples"] is not None:
            merged[f"free_{k}"] = {
                "name":    free_scenarios[k]["name"],
                "color":   C_ORANGE if "om" not in k else C_VERMILION,
                "samples": free_scenarios[k]["samples"][:, :7],
            }

    fig12 = plot_dspl_corner(
        merged, truth, fixed_params,
        scenarios_to_plot=[
            "fixed_lsst_y10","free_lsst_y10",
            "fixed_lsst_y10_om_prior","free_lsst_y10_om_prior",
        ],
        custom_ranges=custom_ranges_7, latex_labels=latex_labels, figsize=(14,14),
        show_multiple_titles=True,
        custom_linestyles_dict={
            "fixed_lsst_y10":          "-",
            "free_lsst_y10":           "--",
            "fixed_lsst_y10_om_prior": "-",
            "free_lsst_y10_om_prior":  "--",
        },
        save_path=f"{FIGURE_DIR}/free_vs_fixed_scatter_lsst_y10_forecast_w0waCDM.pdf",
    )
    plt.close(fig12); print("  → Fig 12 saved")
else:
    print("  [skip] free-scatter posteriors not found")


# ── Cell 16: Summary ─────────────────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"All figures written to:  {FIGURE_DIR}/")
print(f"{'─'*60}")
for fname in sorted(f for f in os.listdir(FIGURE_DIR) if f.endswith((".pdf",".png"))):
    size_kb = os.path.getsize(os.path.join(FIGURE_DIR, fname)) / 1024
    print(f"  {fname:<70}  {size_kb:6.0f} KB")